[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/25_flash_attention_interview.ipynb)

# 🔴 Solution: Flash Attention (Tiled)（面试版）

## 📋 题目描述

**难度：** Hard

实现分块（Flash）注意力与在线 softmax。

Flash 注意力将 Q/K/V 分块处理以减少内存使用，利用在线 softmax 技巧在分块间保持数值正确性。

**签名:** `flash_attention(Q, K, V, block_size=32) -> Tensor`

**参数:**
- `Q`, `K`, `V` — 输入张量 (B, S, D)
- `block_size` — 分块大小

**返回:** 注意力输出 (B, S, D)，与标准注意力数值一致

**约束:**
- 必须与标准 softmax 注意力数值匹配
- 处理非对齐序列长度（S 不能被 block_size 整除）
- 结果不受 block_size 选择影响

**提示：** 分块处理 Q。对每个 Q 块，遍历 K/V 块：
1. `block_max = scores.max(dim=-1)` → `new_max = max(row_max, block_max)`
2. `correction = exp(row_max - new_max)` → 重新缩放 `acc` 和 `row_sum`
3. `acc += exp(scores - new_max) @ V_block`
4. `output = acc / row_sum`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

def flash_attention(Q, K, V, block_size=32):
    B, S, D = Q.shape
    output = torch.zeros_like(Q)

    # 外层：遍历 Q 块
    for i in range(0, S, block_size):
        qi = Q[:, i:i+block_size]
        bs_q = qi.shape[1]

        # 在线状态：每个 query 位置维护自己的 max 和 sum
        row_max = torch.full((B, bs_q, 1), float('-inf'), device=Q.device)
        row_sum = torch.zeros(B, bs_q, 1, device=Q.device)
        acc = torch.zeros(B, bs_q, D, device=Q.device)

        # 内层：遍历 K/V 块
        for j in range(0, S, block_size):
            kj = K[:, j:j+block_size]
            vj = V[:, j:j+block_size]

            # 分块注意力分数
            scores = torch.bmm(qi, kj.transpose(1, 2)) / math.sqrt(D)

            # ---- 在线 softmax 核心 ----
            block_max = scores.max(dim=-1, keepdim=True).values
            new_max = torch.maximum(row_max, block_max)

            # 修正系数：之前累加的值需要按 exp(旧max - 新max) 缩放
            correction = torch.exp(row_max - new_max)

            # 当前块的 exp(scores - new_max)
            exp_scores = torch.exp(scores - new_max)

            # 更新累加器（先修正旧值，再加新值）
            acc = acc * correction + torch.bmm(exp_scores, vj)
            row_sum = row_sum * correction + exp_scores.sum(dim=-1, keepdim=True)
            row_max = new_max

        # 归一化得到最终输出
        output[:, i:i+block_size] = acc / row_sum

    return output

# 面试核心追问：
# Q: Flash Attention 的内存优势？
# A: 标准注意力需要存储 S×S 的注意力矩阵 → O(S²) 内存
#    Flash Attention 分块处理，只维护 O(S) 的累加器
#    实际中 GPU 的 SRAM（L2 cache）能装下小块，避免反复读写 HBM
# Q: 在线 softmax 的原理？
# A: softmax(x) = exp(x-max) / sum(exp(x-max))
#    分块时 max 可能更新，需要用 exp(旧max-新max) 修正之前的累加结果
# Q: 结果为什么和标准注意力完全一样？
# A: 在线 softmax 是精确的（不是近似），只是计算顺序不同


In [ ]:
Q = torch.randn(1, 16, 32)
K = torch.randn(1, 16, 32)
V = torch.randn(1, 16, 32)
out_flash = flash_attention(Q, K, V, block_size=4)
out_std = torch.softmax(Q @ K.T / math.sqrt(32), dim=-1) @ V
print(f'Match: {torch.allclose(out_flash, out_std, atol=1e-5)}')


In [ ]:
from torch_judge import check
check('flash_attention')


## 📝 核心思路总结

1. **分块处理**：Q 和 K/V 都按 block_size 分块，避免存储完整 S×S 注意力矩阵
2. **在线 softmax**：维护 row_max 和 row_sum，每块后用修正系数更新
3. **修正系数**：`correction = exp(旧max - 新max)`，确保之前累加值的数值正确性
4. **内存优势**：O(S) vs O(S²)，对长序列至关重要
